[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/059.%20The%20Warehouse%20Layout%20Design%20Problem/P59-Tier-5_executed.ipynb)

---


# 59. The Warehouse Layout Design Problem

## Tier 5: Integrated Digital Twin (System-of-Systems Simulation)

### Purpose, inputs, and outputs

This notebook represents a warehouse digital-twin architecture and uses a small, rule-based scenario evaluation to illustrate how changing a layout can affect estimated operating measures. Its inputs are a synthetic warehouse configuration, directed material-flow intensities, and seasonal demand multipliers. Its outputs are simulated sensor and performance records, estimates for a fixed set of layout scenarios, and an illustrative financial analysis.

The layout method used here is deliberately explicit: construct five candidate layouts using fixed rules, calculate each candidate's weighted Manhattan flow distance, estimate operating measures from that distance and the demand level, compute a composite score, and select the candidate with the highest score. This is a scenario-based heuristic evaluation, not a mathematical programming solver or a learned optimization policy.

### Scope and modeling assumptions

- The warehouse grid is a 5-by-4 set of discrete locations. A layout assigns each named area to one location index; the corresponding grid coordinate is used to calculate travel distance.
- Flow intensities are directed weights. Distance is Manhattan distance, so travel is measured along horizontal and vertical grid steps rather than straight-line distance.
- Sensor readings, analytics, external-system connections, and operational changes are simulated. The notebook does not connect to physical sensors, enterprise systems, or trained forecasting or reinforcement-learning models.
- The four architecture layers are software representations of system roles. Their presence in the code does not establish real-time integration, predictive accuracy, or physical-to-virtual synchronization.
- The five candidate layouts are illustrative rule-generated scenarios. The implementation does not evaluate fifteen scenarios, relocate individual SKUs, or enforce real-world aisle, safety, capacity, or equipment constraints beyond assigning distinct locations to the named areas.
- The performance equations and financial assumptions are teaching proxies, not empirically calibrated warehouse models or independently validated business results. Their estimates should not be interpreted as measured outcomes.

### Architecture and workflow

The modeled information flow is **Physical Assets → Connectivity → Data Processing → Application**. Sensor-like records pass through the first three layers; the application layer generates simulated operational-change events. Separately, the seasonal analysis constructs and evaluates layout scenarios using the explicit flow-distance and score calculations described below.

The main example uses a 200,000-square-foot warehouse, 15,000 daily orders, 500 sensors, and 50 RFID readers. The corresponding estimate of 1.5 million daily data points is calculated as 100 points per order; it is a configuration assumption, not a measured processing rate. The seasonal examples use demand multipliers of 3.4, 2.5, and 0.6.

### Layout evaluation: intuition and mathematical model

A high-flow pair of areas contributes more to the layout's travel burden than a low-flow pair at the same distance. For candidate layout \(L\), the notebook calculates the weighted flow distance

$$D(L)=\sum_{(i,j)\in F} f_{ij}\,d(L_i,L_j),\qquad
d(L_i,L_j)=|x_{L_i}-x_{L_j}|+|y_{L_i}-y_{L_j}|.$$

Here, \(F\) is the set of directed area-to-area flows, \(f_{ij}\) is the flow intensity from area \(i\) to area \(j\), \(L_i\) is the location index assigned to area \(i\), and \((x_k,y_k)\) is the grid coordinate for location index \(k\). In the code, these quantities are represented by `config['flows']`, `layout`, `config['locations']`, and `calculate_layout_performance`.

The flow distance is converted into illustrative operating measures. Let \(q\) be daily orders and \(r=D(L)/100000\). The notebook uses

$$t(L)=8.5\left(1+0.1r\right),\qquad
u(q)=\min\left(95,70+10\frac{q}{15000}\right),$$

$$e(L)=\min\left(95,80+0.05(100-r)\right),\qquad
S(L,q)=100-5t(L)+0.3u(q)+0.2e(L).$$

Here, \(t\) is estimated pick time in minutes, \(u\) is estimated space utilization in percent, \(e\) is estimated labor efficiency in percent, and \(S\) is a dimensionless composite score. The constants are illustrative scaling choices; they are not fitted parameters. The scenario rule selects \(L^*\in\arg\max_{L\in\mathcal{L}}S(L,q)\), where \(\mathcal{L}\) is the set of five generated candidates. If scores tie, Python's `max` returns the first candidate in insertion order.

This is a finite comparison of five candidates, not a search over all possible warehouse layouts. If there are \(m\) candidate layouts and \(p=|F|\) directed flows, evaluating all candidates takes \(O(mp)\) flow-distance operations, in addition to the cost of generating the candidates. The Python functions `simulate_layout_scenario`, `calculate_layout_performance`, and `analyze_seasonal_scenario` implement this sequence.

### Reading the results

A positive pick-time reduction means the selected scenario's estimated pick time is lower than the simulated baseline average; a negative value means it is higher. Space-utilization and labor-efficiency changes are differences in percentage points, not relative percentage changes. Seasonal comparisons use a baseline average from the simulated records when those records exist. As a result, changing the random seed or the generated simulation data can change the comparisons. No output should be treated as a real operational measurement or a guaranteed business outcome.

In [ ]:
import random
import time
from collections import defaultdict
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds make the synthetic examples reproducible when cells run in order.
np.random.seed(42)
random.seed(42)

plt.style.use("default")

In [ ]:
class WarehouseDigitalTwin:
    """Coordinate simulated warehouse layers and scenario evaluation."""

    def __init__(self, warehouse_config):
        self.config = warehouse_config
        self.physical_layer = PhysicalAssetLayer(self.config)
        self.connectivity_layer = ConnectivityLayer(self.config)
        self.data_layer = DataProcessingLayer(self.config)
        self.application_layer = ApplicationLayer(self.config)

        self.current_time = datetime(2024, 1, 1)
        self.simulation_data = []
        self.performance_metrics = defaultdict(list)
        self.active_scenarios = {}
        self.optimal_layout = None

    def initialize_systems(self):
        """Initialize the simulated layers and integration placeholders."""
        print("Initializing digital twin architecture...")
        self.physical_layer.deploy_sensors()
        self.physical_layer.setup_rfid_readers()
        self.physical_layer.setup_equipment()
        self.connectivity_layer.establish_connections()
        self.connectivity_layer.configure_edge_computing()
        self.data_layer.setup_analytics_engine()
        self.data_layer.load_predictive_models()
        self.application_layer.setup_optimization_engine()
        self.application_layer.initialize_dashboard()
        self.connect_external_systems()
        print("Digital twin simulation initialized.")

    def connect_external_systems(self):
        """Create simulated integration objects for enterprise systems."""
        self.external_systems = {
            "ERP": ERPIntegration(),
            "WMS": WMSIntegration(),
            "TMS": TMSIntegration(),
            "CRM": CRMIntegration(),
        }
        for system_name, system in self.external_systems.items():
            system.establish_connection()
            print(f"Initialized simulated {system_name} integration.")

    def run_simulation(self, duration_days=30, time_step_minutes=15):
        """Generate synthetic records at regular simulation intervals."""
        if duration_days <= 0 or time_step_minutes <= 0:
            raise ValueError("Duration and time step must be positive.")

        print(
            f"Running {duration_days}-day simulation with "
            f"{time_step_minutes}-minute intervals..."
        )
        time_steps = int(duration_days * 24 * 60 / time_step_minutes)
        progress_interval = max(1, time_steps // 10)

        for step in range(time_steps):
            self.current_time += timedelta(minutes=time_step_minutes)
            sensor_data = self.physical_layer.collect_sensor_data(self.current_time)
            processed_data = self.connectivity_layer.transmit_data(sensor_data)
            analytics_results = self.data_layer.process_data(
                processed_data, self.current_time
            )
            optimization_results = self.application_layer.optimize_layout(
                analytics_results, self.current_time
            )

            self.simulation_data.append(
                {
                    "timestamp": self.current_time,
                    "sensor_data": sensor_data,
                    "analytics": analytics_results,
                    "optimization": optimization_results,
                }
            )
            self.update_performance_metrics(
                analytics_results, optimization_results
            )

            if step % progress_interval == 0:
                progress = 100 * step / time_steps
                print(f"Simulation progress: {progress:.0f}%")

        print("Simulation completed.")

    def update_performance_metrics(self, analytics, optimization):
        """Store each synthetic metric for later summaries and plots."""
        self.performance_metrics["pick_time"].append(
            analytics.get("avg_pick_time", 0)
        )
        self.performance_metrics["space_utilization"].append(
            analytics.get("space_utilization", 0)
        )
        self.performance_metrics["labor_efficiency"].append(
            analytics.get("labor_efficiency", 0)
        )
        self.performance_metrics["forecast_accuracy"].append(
            analytics.get("forecast_accuracy", 0)
        )
        self.performance_metrics["layout_changes"].append(
            optimization.get("layout_changes", 0)
        )

    def analyze_seasonal_scenario(
        self, scenario_name, demand_multiplier, duration_days=7
    ):
        """Evaluate the fixed candidate layouts under a demand scenario."""
        if demand_multiplier <= 0 or duration_days <= 0:
            raise ValueError("Demand multiplier and duration must be positive.")

        print(f"\nAnalyzing seasonal scenario: {scenario_name}")
        print(f"Demand multiplier: {demand_multiplier}x")
        print(f"Duration: {duration_days} days")

        # The baseline is an average of recent synthetic observations when available.
        baseline_metrics = {
            "pick_time": (
                float(np.mean(self.performance_metrics["pick_time"][-100:]))
                if self.performance_metrics["pick_time"]
                else 8.5
            ),
            "space_utilization": (
                float(np.mean(self.performance_metrics["space_utilization"][-100:]))
                if self.performance_metrics["space_utilization"]
                else 70.0
            ),
            "labor_efficiency": (
                float(np.mean(self.performance_metrics["labor_efficiency"][-100:]))
                if self.performance_metrics["labor_efficiency"]
                else 80.0
            ),
        }

        seasonal_config = self.config.copy()
        seasonal_config["daily_orders"] = int(
            self.config["daily_orders"] * demand_multiplier
        )
        layout_scenarios = self.generate_layout_scenarios(
            seasonal_config, scenario_name
        )

        scenario_results = {}
        for scenario_id, layout_config in layout_scenarios.items():
            print(f"  Evaluating scenario {scenario_id}...")
            scenario_performance = self.simulate_layout_scenario(
                layout_config, seasonal_config, duration_days
            )
            scenario_results[scenario_id] = {
                "layout": layout_config,
                "performance": scenario_performance,
                "improvement": {
                    "pick_time_reduction": (
                        (baseline_metrics["pick_time"]
                         - scenario_performance["avg_pick_time"])
                        / baseline_metrics["pick_time"]
                        * 100
                    ),
                    "space_utilization_improvement": (
                        scenario_performance["space_utilization"]
                        - baseline_metrics["space_utilization"]
                    ),
                    "labor_efficiency_improvement": (
                        scenario_performance["labor_efficiency"]
                        - baseline_metrics["labor_efficiency"]
                    ),
                },
            }

        optimal_scenario = max(
            scenario_results.items(),
            key=lambda item: item[1]["performance"]["overall_score"],
        )
        selected_result = optimal_scenario[1]
        selected_result["baseline_metrics"] = baseline_metrics
        selected_result["scenario_id"] = optimal_scenario[0]

        print(f"\nSelected scenario for {scenario_name}: {optimal_scenario[0]}")
        print("Estimated changes relative to the simulated baseline:")
        improvements = selected_result["improvement"]
        print(f"  Pick-time reduction: {improvements['pick_time_reduction']:.1f}%")
        print(
            "  Space-utilization change: "
            f"{improvements['space_utilization_improvement']:.1f} percentage points"
        )
        print(
            "  Labor-efficiency change: "
            f"{improvements['labor_efficiency_improvement']:.1f} percentage points"
        )
        return selected_result

    def generate_layout_scenarios(self, config, scenario_name):
        """Create the five rule-based layout candidates used in this example."""
        return {
            "baseline": config["current_layout"].copy(),
            "high_demand": self.optimize_for_high_demand(config),
            "space_efficient": self.optimize_for_space_efficiency(config),
            "balanced": self.optimize_balanced(config),
            "automation_ready": self.optimize_for_automation(config),
        }

    def optimize_for_high_demand(self, config):
        """Build a rule-based candidate that places fast storage near packing."""
        layout = config["current_layout"].copy()
        layout["Returns"] = 17
        layout["Fast_Storage"] = 8
        layout["Medium_Storage"] = 9
        layout["Packing"] = 13
        layout["Shipping"] = 18
        return layout

    def optimize_for_space_efficiency(self, config):
        """Build a candidate that groups the storage areas on the grid."""
        layout = config["current_layout"].copy()
        layout["Slow_Storage"] = 0
        layout["Medium_Storage"] = 4
        layout["Fast_Storage"] = 9
        return layout

    def optimize_balanced(self, config):
        """Build a fixed candidate with a balanced sequence of area locations."""
        layout = config["current_layout"].copy()
        layout["Receiving"] = 19
        layout["Fast_Storage"] = 14
        layout["Medium_Storage"] = 9
        layout["Slow_Storage"] = 4
        layout["Packing"] = 13
        layout["Shipping"] = 8
        return layout

    def optimize_for_automation(self, config):
        """Build a fixed candidate representing an automation-oriented layout."""
        layout = config["current_layout"].copy()
        layout["Receiving"] = 19
        layout["Fast_Storage"] = 14
        layout["Packing"] = 8
        layout["Shipping"] = 3
        return layout

    def simulate_layout_scenario(self, layout, config, duration_days):
        """Estimate scenario measures from flow distance and demand."""
        if set(layout) != set(config["current_layout"]):
            raise ValueError("Every named area must have exactly one location.")
        if len(set(layout.values())) != len(layout):
            raise ValueError("Each named area must occupy a distinct location.")
        if any(index < 0 or index >= len(config["locations"])
               for index in layout.values()):
            raise ValueError("A layout location index is outside the grid.")

        total_flow_distance = self.calculate_layout_performance(layout, config)
        distance_factor = total_flow_distance / 100000
        avg_pick_time = 8.5 * (1 + distance_factor * 0.1)
        space_utilization = min(
            95, 70 + (config["daily_orders"] / 15000) * 10
        )
        labor_efficiency = min(95, 80 + (100 - distance_factor) * 0.05)
        overall_score = (
            (100 - avg_pick_time * 5)
            + space_utilization * 0.3
            + labor_efficiency * 0.2
        )

        return {
            "avg_pick_time": avg_pick_time,
            "space_utilization": space_utilization,
            "labor_efficiency": labor_efficiency,
            "total_flow_distance": total_flow_distance,
            "overall_score": overall_score,
        }

    def calculate_layout_performance(self, layout, config):
        """Sum directed flow intensity times Manhattan distance."""
        total_distance = 0.0
        for (area_from, area_to), flow_intensity in config["flows"].items():
            if flow_intensity > 0:
                loc_from = config["locations"][layout[area_from]]
                loc_to = config["locations"][layout[area_to]]
                distance = (
                    abs(loc_from[0] - loc_to[0])
                    + abs(loc_from[1] - loc_to[1])
                )
                total_distance += flow_intensity * distance
        return total_distance

    def generate_comprehensive_report(self):
        """Print summaries of the synthetic records collected so far."""
        print("\n" + "=" * 72)
        print("SIMULATED WAREHOUSE DIGITAL TWIN REPORT")
        print("=" * 72)
        print("\nSYSTEM CONFIGURATION:")
        print(f"  Simulation records: {len(self.simulation_data)}")
        print(f"  Configured IoT sensors: {self.config['num_sensors']}")
        print(f"  Configured RFID readers: {self.config['num_rfid_readers']}")
        print(f"  Simulated system integrations: {len(self.external_systems)}")
        print(
            "  Assumed daily data points: "
            f"{self.config['daily_orders'] * 100:,}"
        )

        if self.performance_metrics["pick_time"]:
            print("\nMEAN SYNTHETIC PERFORMANCE METRICS:")
            print(
                "  Average pick time: "
                f"{np.mean(self.performance_metrics['pick_time']):.2f} minutes"
            )
            print(
                "  Space utilization: "
                f"{np.mean(self.performance_metrics['space_utilization']):.1f}%"
            )
            print(
                "  Labor efficiency: "
                f"{np.mean(self.performance_metrics['labor_efficiency']):.1f}%"
            )
            print(
                "  Forecast accuracy: "
                f"{np.mean(self.performance_metrics['forecast_accuracy']):.1f}%"
            )

        if self.performance_metrics["layout_changes"]:
            total_changes = sum(self.performance_metrics["layout_changes"])
            records = len(self.performance_metrics["layout_changes"])
            changes_per_day = total_changes / records * 24
            print("\nSIMULATED LAYOUT-CHANGE EVENTS:")
            print(f"  Total events: {total_changes}")
            print(f"  Events per simulated day: {changes_per_day:.1f}")
        print("\n" + "=" * 72)

In [ ]:
class PhysicalAssetLayer:
    """Generate synthetic sensor, RFID, and equipment records."""

    def __init__(self, config):
        self.config = config
        self.sensors = []
        self.rfid_readers = []
        self.equipment = []

    def deploy_sensors(self):
        sensor_types = [
            "picker_location",
            "inventory_level",
            "equipment_status",
            "environmental",
            "traffic_flow",
        ]
        sensors_per_type = self.config["num_sensors"] // len(sensor_types)
        for sensor_type in sensor_types:
            for index in range(sensors_per_type):
                self.sensors.append(
                    {
                        "id": f"{sensor_type}_{index}",
                        "type": sensor_type,
                        "location": (random.randint(1, 5), random.randint(1, 4)),
                        "status": "active",
                        "data_rate_hz": random.uniform(0.5, 2.0),
                    }
                )
        print(f"Generated {len(self.sensors)} simulated sensor definitions.")

    def setup_rfid_readers(self):
        for index in range(self.config["num_rfid_readers"]):
            self.rfid_readers.append(
                {
                    "id": f"rfid_{index}",
                    "location": (random.randint(1, 5), random.randint(1, 4)),
                    "range_m": 10,
                    "read_rate_tags_per_second": random.uniform(10, 50),
                }
            )
        print(f"Generated {len(self.rfid_readers)} simulated RFID readers.")

    def setup_equipment(self):
        for equipment_type in ["conveyor", "sorter", "picker", "packer"]:
            for index in range(5):
                self.equipment.append(
                    {
                        "id": f"{equipment_type}_{index}",
                        "type": equipment_type,
                        "status": "operational",
                        "utilization": random.uniform(0.3, 0.9),
                        "maintenance_due_days": random.randint(1, 30),
                    }
                )
        print(f"Generated {len(self.equipment)} simulated equipment records.")

    def collect_sensor_data(self, timestamp):
        sensor_data = {
            "timestamp": timestamp,
            "picker_locations": [],
            "inventory_levels": [],
            "equipment_status": [],
            "environmental": [],
            "traffic_flow": [],
        }
        for sensor in self.sensors:
            if sensor["type"] == "picker_location":
                sensor_data["picker_locations"].append(
                    {
                        "sensor_id": sensor["id"],
                        "picker_id": random.randint(1, 20),
                        "location": sensor["location"],
                        "speed": random.uniform(0.5, 2.0),
                        "timestamp": timestamp,
                    }
                )
            elif sensor["type"] == "inventory_level":
                sensor_data["inventory_levels"].append(
                    {
                        "sensor_id": sensor["id"],
                        "sku": f"SKU_{random.randint(1000, 9999)}",
                        "quantity": random.randint(0, 1000),
                        "location": sensor["location"],
                        "timestamp": timestamp,
                    }
                )
        return sensor_data


class ConnectivityLayer:
    """Represent simulated edge nodes and data transmission."""

    def __init__(self, config):
        self.config = config
        self.edge_nodes = []
        self.data_streams = []

    def establish_connections(self):
        for index in range(4):
            self.edge_nodes.append(
                {
                    "id": f"edge_node_{index}",
                    "location": (index + 1, 3),
                    "capacity_gb": 1000,
                    "processing_power_gflops": 50,
                    "latency_ms": random.uniform(1, 5),
                }
            )
        print(f"Generated {len(self.edge_nodes)} simulated edge nodes.")

    def configure_edge_computing(self):
        for node in self.edge_nodes:
            node["active_streams"] = 0
            node["data_processed"] = 0
        print("Configured simulated edge-processing state.")

    def transmit_data(self, sensor_data):
        return {
            "timestamp": sensor_data["timestamp"],
            "data_volume_kb": self.calculate_data_volume(sensor_data),
            "processing_latency_ms": random.uniform(10, 50),
            "transmission_success_rate": random.uniform(0.95, 0.99),
        }

    def calculate_data_volume(self, sensor_data):
        total_data_points = sum(
            len(value) for value in sensor_data.values()
            if isinstance(value, list)
        )
        return total_data_points * 1024


class DataProcessingLayer:
    """Produce illustrative analytics values rather than trained predictions."""

    def __init__(self, config):
        self.config = config
        self.analytics_engine = None
        self.ml_models = {}

    def setup_analytics_engine(self):
        self.analytics_engine = {
            "streaming_analytics": True,
            "batch_processing": True,
            "real_time_alerts": True,
            "historical_analysis": True,
        }
        print("Configured simulated analytics capabilities.")

    def load_predictive_models(self):
        # These entries describe placeholders; no model is trained or executed.
        self.ml_models = {
            "demand_forecast": {
                "model_type": "illustrative forecast placeholder",
                "nominal_accuracy_fraction": 0.94,
                "update_frequency": "hourly",
            },
            "layout_scenario_analysis": {
                "model_type": "fixed rule-based candidates",
                "candidate_count": 5,
            },
            "anomaly_detection": {
                "model_type": "illustrative placeholder",
                "nominal_false_positive_rate": 0.02,
                "nominal_detection_latency_seconds": 30,
            },
        }
        print(f"Configured {len(self.ml_models)} model placeholders.")

    def process_data(self, processed_data, timestamp):
        return {
            "timestamp": timestamp,
            "avg_pick_time": random.uniform(6.0, 10.0),
            "space_utilization": random.uniform(65, 95),
            "labor_efficiency": random.uniform(70, 95),
            "forecast_accuracy": random.uniform(85, 98),
            "anomalies_detected": random.randint(0, 5),
            "alerts_generated": random.randint(0, 3),
        }


class ApplicationLayer:
    """Represent simulated monitoring and operational-change events."""

    def __init__(self, config):
        self.config = config
        self.optimization_engine = None
        self.dashboard = None

    def setup_optimization_engine(self):
        self.optimization_engine = {
            "method": "fixed scenario evaluation and synthetic event generation",
            "objective": "illustrative composite performance score",
            "scenario_analysis": True,
        }
        print("Configured the illustrative scenario-analysis interface.")

    def initialize_dashboard(self):
        self.dashboard = {
            "real_time_metrics": True,
            "layout_visualization": True,
            "performance_trends": True,
            "alert_management": True,
        }
        print("Configured a simulated monitoring dashboard.")

    def optimize_layout(self, analytics_results, timestamp):
        # This creates synthetic event records; it does not choose a layout.
        return {
            "timestamp": timestamp,
            "layout_changes": random.randint(0, 3),
            "expected_improvement_percent": random.uniform(5, 15),
            "implementation_time_minutes": random.uniform(30, 120),
        }


class ERPIntegration:
    def establish_connection(self):
        print("ERP integration placeholder initialized.")


class WMSIntegration:
    def establish_connection(self):
        print("WMS integration placeholder initialized.")


class TMSIntegration:
    def establish_connection(self):
        print("TMS integration placeholder initialized.")


class CRMIntegration:
    def establish_connection(self):
        print("CRM integration placeholder initialized.")

## Configure the warehouse

The location list defines the grid coordinates in index order: index 0 is at \((1,4)\), index 1 at \((2,4)\), and so on through index 19 at \((5,1)\). Each flow key is an ordered pair, so the recorded flow from receiving to storage is distinct from a flow in the reverse direction. The layout checks in the scenario evaluator ensure that each named area has one valid, distinct grid location.

In [ ]:
warehouse_config = {
    "warehouse_size": 200000,
    "daily_orders": 15000,
    "num_sensors": 500,
    "num_rfid_readers": 50,
    "locations": [
        (1, 4), (2, 4), (3, 4), (4, 4), (5, 4),
        (1, 3), (2, 3), (3, 3), (4, 3), (5, 3),
        (1, 2), (2, 2), (3, 2), (4, 2), (5, 2),
        (1, 1), (2, 1), (3, 1), (4, 1), (5, 1),
    ],
    "current_layout": {
        "Receiving": 19,
        "Returns": 18,
        "Fast_Storage": 14,
        "Medium_Storage": 9,
        "Slow_Storage": 4,
        "Packing": 13,
        "Shipping": 8,
    },
    "flows": {
        ("Receiving", "Returns"): 50,
        ("Receiving", "Fast_Storage"): 480,
        ("Receiving", "Medium_Storage"): 280,
        ("Receiving", "Slow_Storage"): 120,
        ("Returns", "Fast_Storage"): 30,
        ("Fast_Storage", "Packing"): 450,
        ("Medium_Storage", "Packing"): 250,
        ("Slow_Storage", "Packing"): 100,
        ("Packing", "Shipping"): 800,
    },
}

assert len(warehouse_config["locations"]) == 20
assert len(set(warehouse_config["locations"])) == 20
assert set(warehouse_config["current_layout"]) == {
    area for flow in warehouse_config["flows"] for area in flow
}

print("ILLUSTRATIVE WAREHOUSE CONFIGURATION")
print("=" * 48)
print(f"Warehouse area: {warehouse_config['warehouse_size']:,} sq ft")
print(f"Daily orders: {warehouse_config['daily_orders']:,}")
print(f"Configured sensors: {warehouse_config['num_sensors']}")
print(f"Configured RFID readers: {warehouse_config['num_rfid_readers']}")
print(
    "Assumed daily data points: "
    f"{warehouse_config['daily_orders'] * 100:,}"
)
print("Simulated integration types: ERP, WMS, TMS, and CRM")

## Initialize the layers and generate baseline records

The baseline run creates synthetic records at hourly intervals. One record is collected per time step, so this run provides a simulated reference sample rather than a warehouse measurement. Its recent averages are used as the comparison baseline in the seasonal analysis.

In [ ]:
digital_twin = WarehouseDigitalTwin(warehouse_config)
digital_twin.initialize_systems()

print("\nRunning the synthetic baseline simulation...")
start_time = time.time()
digital_twin.run_simulation(duration_days=30, time_step_minutes=60)
elapsed_seconds = time.time() - start_time

print(f"Simulation runtime: {elapsed_seconds:.2f} seconds")
print(f"Synthetic records collected: {len(digital_twin.simulation_data)}")

## Evaluate seasonal layout scenarios

For each demand multiplier, the notebook builds the same five rule-based candidates. Demand affects the space-utilization proxy; each candidate's assigned locations affect its weighted flow distance, estimated pick time, labor-efficiency proxy, and composite score. The selected candidate is the maximum-score member of this finite set, not a globally optimal warehouse layout.

In [ ]:
print("\n" + "=" * 72)
print("SEASONAL SCENARIO ANALYSIS")
print("=" * 72)

back_to_school_results = digital_twin.analyze_seasonal_scenario(
    "Back-to-School Season",
    demand_multiplier=3.4,
    duration_days=7,
)

holiday_results = digital_twin.analyze_seasonal_scenario(
    "Holiday Season",
    demand_multiplier=2.5,
    duration_days=14,
)

slow_season_results = digital_twin.analyze_seasonal_scenario(
    "Slow Season",
    demand_multiplier=0.6,
    duration_days=10,
)

digital_twin.generate_comprehensive_report()

## Visualize simulated performance and scenario comparisons

The first panel overlays measures with different units; it is useful for observing trends, but the shared vertical axis does not make their magnitudes directly comparable. The seasonal panel uses separate vertical axes for pick-time reduction and space-utilization change. The final panel shows an illustrative within-day data-volume profile and reference capacity lines; the profile is synthetic and is not inferred from the simulation records.

In [ ]:
def visualize_digital_twin_performance(digital_twin):
    """Plot synthetic metrics, architecture layers, and scenario comparisons."""
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    ax1, ax2, ax3, ax4 = axes.flat

    ax1.set_title("Synthetic Performance Metrics")
    if digital_twin.performance_metrics["pick_time"]:
        time_points = range(len(digital_twin.performance_metrics["pick_time"]))
        ax1.plot(
            time_points,
            digital_twin.performance_metrics["pick_time"],
            label="Pick time (minutes)",
        )
        ax1.plot(
            time_points,
            digital_twin.performance_metrics["space_utilization"],
            label="Space utilization (percent)",
        )
        ax1.plot(
            time_points,
            digital_twin.performance_metrics["labor_efficiency"],
            label="Labor efficiency (percent)",
        )
        ax1.set_xlabel("Simulation time step (hourly)")
        ax1.set_ylabel("Metric value (different units)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

    ax2.set_title("Digital Twin Architecture Layers")
    layer_names = ["Physical assets", "Connectivity", "Data processing", "Application"]
    layer_components = [
        "500 sensors\n50 RFID readers\n20 equipment units",
        "4 edge nodes\nSimulated transmission",
        "Analytics placeholders\nSynthetic metrics",
        "Scenario evaluation\nMonitoring interface",
    ]
    y_positions = np.arange(len(layer_names))
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(layer_names)))
    ax2.barh(y_positions, np.ones(len(layer_names)), color=colors, alpha=0.75)
    for position, name, components in zip(
        y_positions, layer_names, layer_components
    ):
        ax2.text(0.5, position, f"{name}\n{components}", ha="center", va="center")
    ax2.set_xlim(0, 1)
    ax2.set_yticks([])
    ax2.set_xticks([])
    ax2.set_xlabel("Layer representation; bar colors identify layers")
    ax2.invert_yaxis()

    ax3.set_title("Seasonal Scenario Estimates")
    season_names = ["Back-to-school", "Holiday", "Slow season"]
    results = [
        back_to_school_results,
        holiday_results,
        slow_season_results,
    ]
    pick_changes = [
        result["improvement"]["pick_time_reduction"] for result in results
    ]
    space_changes = [
        result["improvement"]["space_utilization_improvement"]
        for result in results
    ]
    x_positions = np.arange(len(season_names))
    width = 0.35
    ax3.bar(
        x_positions - width / 2,
        pick_changes,
        width,
        label="Pick-time reduction (percent)",
        color="steelblue",
        alpha=0.75,
    )
    ax3_twin = ax3.twinx()
    ax3_twin.bar(
        x_positions + width / 2,
        space_changes,
        width,
        label="Space-utilization change (percentage points)",
        color="darkorange",
        alpha=0.75,
    )
    ax3.set_xlabel("Demand scenario")
    ax3.set_ylabel("Pick-time reduction (percent)")
    ax3_twin.set_ylabel("Space-utilization change (percentage points)")
    ax3.set_xticks(x_positions)
    ax3.set_xticklabels(season_names)
    ax3.legend(loc="upper left")
    ax3_twin.legend(loc="upper right")
    ax3.grid(True, alpha=0.3)

    ax4.set_title("Illustrative Daily Data-Volume Profile")
    hours = np.arange(24)
    base_volume = warehouse_config["daily_orders"] * 100 / 24
    data_volumes = []
    for hour in hours:
        if 6 <= hour <= 10 or 14 <= hour <= 18:
            multiplier = random.uniform(1.5, 2.5)
        elif 11 <= hour <= 13:
            multiplier = random.uniform(1.2, 1.8)
        else:
            multiplier = random.uniform(0.3, 0.8)
        data_volumes.append(base_volume * multiplier)

    ax4.fill_between(hours, data_volumes, alpha=0.25, color="green")
    ax4.plot(hours, data_volumes, color="green", label="Illustrative data volume")
    reference_capacity = max(data_volumes) * 1.2
    ax4.axhline(
        reference_capacity * 0.8,
        color="red",
        linestyle="--",
        label="80% of reference capacity",
    )
    ax4.axhline(
        reference_capacity * 0.9,
        color="orange",
        linestyle="--",
        label="90% of reference capacity",
    )
    ax4.set_xlabel("Hour of day")
    ax4.set_ylabel("Assumed data points per hour")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_digital_twin_performance(digital_twin)

## Illustrative economic impact and ROI

This calculation converts the selected back-to-school scenario's estimated differences into monetary values using explicit assumptions. The pick-time calculation uses the same simulated baseline average that defined the reported reduction. Space-utilization change is measured in percentage points, so multiplying facility area by that change divided by 100 gives a proxy for additional effective capacity.

The resulting annual values are scenario-based estimates, not audited savings. In particular, valuing effective capacity at a fixed annual amount per square foot does not establish that the capacity can be monetized. A non-positive estimated annual benefit has no positive payback period under this simple calculation.

In [ ]:
print("\n" + "=" * 72)
print("ILLUSTRATIVE ECONOMIC IMPACT AND ROI")
print("=" * 72)

improvements = back_to_school_results["improvement"]
baseline_metrics = back_to_school_results["baseline_metrics"]
pick_time_reduction = improvements["pick_time_reduction"]
space_utilization_change = improvements["space_utilization_improvement"]
labor_efficiency_change = improvements["labor_efficiency_improvement"]
baseline_pick_time = baseline_metrics["pick_time"]

print("ESTIMATED CHANGES FOR THE BACK-TO-SCHOOL SCENARIO:")
print(f"  Pick-time reduction: {pick_time_reduction:.1f}%")
print(
    "  Space-utilization change: "
    f"{space_utilization_change:.1f} percentage points"
)
print(
    "  Labor-efficiency change: "
    f"{labor_efficiency_change:.1f} percentage points"
)

daily_orders = warehouse_config["daily_orders"]
labor_cost_per_hour = 25
operating_days_per_year = 365
minutes_saved_per_order = baseline_pick_time * pick_time_reduction / 100
total_minutes_saved_daily = daily_orders * minutes_saved_per_order
total_hours_saved_daily = total_minutes_saved_daily / 60
daily_labor_savings = total_hours_saved_daily * labor_cost_per_hour
annual_labor_savings = daily_labor_savings * operating_days_per_year

print("\nLABOR-VALUE PROXY:")
print(f"  Daily orders: {daily_orders:,}")
print(f"  Baseline pick time: {baseline_pick_time:.2f} minutes per order")
print(f"  Estimated time change: {minutes_saved_per_order:.2f} minutes per order")
print(
    f"  Estimated daily time change: {total_minutes_saved_daily:,.0f} minutes "
    f"({total_hours_saved_daily:,.1f} hours)"
)
print(f"  Estimated daily labor value: ${daily_labor_savings:,.0f}")
print(f"  Estimated annual labor value: ${annual_labor_savings:,.0f}")

additional_capacity_sq_ft = (
    warehouse_config["warehouse_size"] * space_utilization_change / 100
)
annual_value_per_sq_ft = 50
annual_space_value = additional_capacity_sq_ft * annual_value_per_sq_ft
print("\nSPACE-CAPACITY VALUE PROXY:")
print(f"  Effective-capacity change: {additional_capacity_sq_ft:,.0f} sq ft")
print(f"  Assumed annual value per sq ft: ${annual_value_per_sq_ft:,.2f}")
print(f"  Estimated annual space value: ${annual_space_value:,.0f}")

total_annual_benefit = annual_labor_savings + annual_space_value
investment_components = {
    "iot_sensors": 500 * 150,
    "rfid_readers": 50 * 500,
    "edge_computing": 200000,
    "software_development": 300000,
    "integration": 150000,
    "training": 50000,
}
total_investment = sum(investment_components.values())
payback_period_months = (
    total_investment / (total_annual_benefit / 12)
    if total_annual_benefit > 0
    else None
)
first_year_roi_percent = (
    (total_annual_benefit - total_investment) / total_investment * 100
)

print(f"\nTOTAL ESTIMATED ANNUAL BENEFIT: ${total_annual_benefit:,.0f}")
print("\nILLUSTRATIVE INVESTMENT:")
print(f"  Total investment: ${total_investment:,.0f}")
for component, cost in investment_components.items():
    print(f"  {component.replace('_', ' ').title()}: ${cost:,.0f}")
if payback_period_months is None:
    print("  Payback period: not finite under the non-positive benefit estimate")
else:
    print(f"  Simple payback period: {payback_period_months:.1f} months")
print(f"  First-year ROI estimate: {first_year_roi_percent:.1f}%")

print("\nCONTEXTUAL BENCHMARK COMPARISON:")
target_pick_time_reduction = 23
target_space_utilization_change = 31
target_payback_months = 14
print(
    f"  Pick-time reduction benchmark: {target_pick_time_reduction}% | "
    f"Scenario estimate: {pick_time_reduction:.1f}% | "
    f"Met: {pick_time_reduction >= target_pick_time_reduction}"
)
print(
    "  Space-utilization benchmark: "
    f"{target_space_utilization_change} percentage points | "
    f"Scenario estimate: {space_utilization_change:.1f} percentage points | "
    f"Met: {space_utilization_change >= target_space_utilization_change}"
)
payback_target_met = (
    payback_period_months is not None
    and payback_period_months <= target_payback_months
)
payback_text = (
    "not finite"
    if payback_period_months is None
    else f"{payback_period_months:.1f} months"
)
print(
    f"  Payback benchmark: {target_payback_months} months | "
    f"Scenario estimate: {payback_text} | Met: {payback_target_met}"
)
print(
    "  These benchmark comparisons are illustrative and do not validate "
    "the synthetic model against operational evidence."
)

## Interpretation, strengths, and limitations

The example connects several system roles that are often considered separately: physical-asset records, communication and edge-processing representations, analytics records, and an application interface. This systems view helps explain why a digital-twin project involves data definitions, interfaces, and operational processes in addition to a layout calculation.

The layout decision itself remains the explicit finite scenario procedure: fixed candidate-generation rules, weighted Manhattan flow distance, proxy performance equations, composite scoring, and maximum-score selection. The architecture simulation does not feed a trained model into this procedure, and the synthetic `layout_changes` records do not implement the selected layout. The computed scenario estimates and ROI therefore demonstrate how assumptions propagate through a model; they do not demonstrate realized performance improvements.

Important omitted real-world factors include aisle and rack geometry, storage capacity, safety and access constraints, congestion, labor schedules, SKU-level demand, handling equipment, relocation costs, uncertainty in forecasts, and the time required to implement a layout. Adding such information would require extending the data and constraints while keeping the scenario-evaluation logic explicit. Larger studies could also evaluate more candidate layouts or more realistic measured data, but those extensions are outside this example.